In [1]:
# If you're on Colab, run this cell first
!pip -q install -U "transformers>=4.43.0" "datasets>=2.20.0" "accelerate>=0.34.0" "evaluate>=0.4.2" scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.4/503.4 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 26.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [2]:
# OPTION A — Upload the JSONL from your computer
from google.colab import files
up = files.upload()  # Choose your english_dataset_1100.jsonl
DATA_PATH = list(up.keys())[0]  # path in /content

# OPTION B — If your file is in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/path/to/english_dataset_1100.jsonl"


Saving english_dataset_1100.jsonl to english_dataset_1100.jsonl


In [3]:
MODEL_NAME = "distilroberta-base"   # good small baseline; you can try "roberta-base", "bert-base-uncased", etc.
MAX_LEN    = 256                    # increase to 512 if your texts are long
BATCH_SIZE = 16
LR         = 2e-5
EPOCHS     = 3                      # start small; raise to 4–6 if needed
WEIGHTED_LOSS = True                # handles class imbalance

# If your JSONL uses different keys, change these:
TEXT_COL  = "text"
LABEL_COL = "label"

SAVE_DIR  = "/content/ai_text_detector"


In [4]:
from datasets import load_dataset
import json, itertools

# Try to infer the columns if user keys differ
def sniff_keys(jsonl_path, n=3):
    keys = set()
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in itertools.islice(f, n):
            obj = json.loads(line)
            keys |= set(obj.keys())
    return keys

print("First few keys found:", sniff_keys(DATA_PATH))

raw_ds = load_dataset("json", data_files=DATA_PATH, split="train")

# Basic checks
print(raw_ds)
print("Example:", raw_ds[0])

# If your file doesn't have these exact names, edit TEXT_COL/LABEL_COL in the config cell above.
assert TEXT_COL in raw_ds.column_names, f"Column '{TEXT_COL}' not found. Available: {raw_ds.column_names}"
assert LABEL_COL in raw_ds.column_names, f"Column '{LABEL_COL}' not found. Available: {raw_ds.column_names}"

# Ensure labels are int and in {0,1}
def fix_label(ex):
    ex[LABEL_COL] = int(ex[LABEL_COL])
    return ex
raw_ds = raw_ds.map(fix_label)

# Train/validation split
ds = raw_ds.train_test_split(test_size=0.15, seed=42)
train_ds = ds["train"]
val_ds   = ds["test"]

# Class distribution
from collections import Counter
print("Train label counts:", Counter(train_ds[LABEL_COL]))
print("Val   label counts:", Counter(val_ds[LABEL_COL]))


First few keys found: {'label', 'text'}


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 1100
})
Example: {'text': 'My mother baked cookies and they smelled amazing.', 'label': 0}


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Train label counts: Counter({0: 478, 1: 457})
Val   label counts: Counter({1: 84, 0: 81})


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COL],
        truncation=True,
        padding=False,       # padding handled by DataCollator
        max_length=MAX_LEN
    )

encoded_train = train_ds.map(tokenize_fn, batched=True, remove_columns=[c for c in train_ds.column_names if c not in [TEXT_COL, LABEL_COL]])
encoded_val   = val_ds.map(tokenize_fn,   batched=True, remove_columns=[c for c in val_ds.column_names   if c not in [TEXT_COL, LABEL_COL]])

# Rename label column to 'labels' to match Trainer expectations
encoded_train = encoded_train.rename_column(LABEL_COL, "labels")
encoded_val   = encoded_val.rename_column(LABEL_COL, "labels")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/935 [00:00<?, ? examples/s]

Map:   0%|          | 0/165 [00:00<?, ? examples/s]

In [6]:
import evaluate
import numpy as np

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1":       f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }


In [7]:
import torch
from collections import Counter

class_weights = None
if WEIGHTED_LOSS:
    counts = Counter(encoded_train["labels"])
    total = sum(counts.values())
    # weight = total / (num_classes * class_count)
    weights = [total / (2 * counts.get(i, 1)) for i in range(2)]
    class_weights = torch.tensor(weights, dtype=torch.float)
    print("Class weights:", class_weights.tolist())


Class weights: [0.9780334830284119, 1.0229759216308594]


In [11]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import torch
from torch import nn

num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# ---- Custom Trainer that supports class weights and HF's new signature ----
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    # NOTE: new signature includes num_items_in_batch
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        # Forward pass (don't pass labels to let us control the loss)
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(
            weight=(self.class_weights.to(logits.device) if self.class_weights is not None else None)
        )
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="/content/ai_text_detector_runs",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    eval_strategy="steps",   # <-- fixed
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=1,
    warmup_ratio=0.06,
)

# Early stopping
from transformers import EarlyStoppingCallback
callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]

# Pass class_weights only if you enabled it earlier
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
    class_weights=class_weights if WEIGHTED_LOSS else None,
)

trainer.train()

# Final eval
metrics = trainer.evaluate()
metrics


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1330843876.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.0002558976411819458,
 'eval_accuracy': 1.0,
 'eval_f1': 1.0,
 'eval_runtime': 15.1672,
 'eval_samples_per_second': 10.879,
 'eval_steps_per_second': 0.725,
 'epoch': 3.0}

In [13]:
# Replace with your folder path
FOLDER_PATH = "/content/ai_text_detector_runs"
ZIP_PATH = "/content/ai_text_detector.zip"

# Zip the folder
!zip -r $ZIP_PATH $FOLDER_PATH

# Download
from google.colab import files
files.download(ZIP_PATH)


  adding: content/ai_text_detector_runs/ (stored 0%)
  adding: content/ai_text_detector_runs/checkpoint-177/ (stored 0%)
  adding: content/ai_text_detector_runs/checkpoint-177/tokenizer.json (deflated 82%)
  adding: content/ai_text_detector_runs/checkpoint-177/tokenizer_config.json (deflated 75%)
  adding: content/ai_text_detector_runs/checkpoint-177/trainer_state.json (deflated 61%)
  adding: content/ai_text_detector_runs/checkpoint-177/training_args.bin (deflated 53%)
  adding: content/ai_text_detector_runs/checkpoint-177/vocab.json (deflated 59%)
  adding: content/ai_text_detector_runs/checkpoint-177/rng_state.pth (deflated 26%)
  adding: content/ai_text_detector_runs/checkpoint-177/scheduler.pt (deflated 62%)
  adding: content/ai_text_detector_runs/checkpoint-177/optimizer.pt (deflated 51%)
  adding: content/ai_text_detector_runs/checkpoint-177/config.json (deflated 49%)
  adding: content/ai_text_detector_runs/checkpoint-177/model.safetensors (deflated 7%)
  adding: content/ai_text

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>